# 🧠 Product Review Intelligence System

### Aspect-Based Sentiment Analysis (ABSA) Dashboard

A complete intelligence system that:

- ✅ Accepts customer reviews
- ✅ Extracts candidate aspects automatically (spaCy POS tagging)
- ✅ Predicts aspect-level sentiment (DeBERTa ABSA model)
- ✅ Calculates confidence scores
- ✅ Visualizes positive and negative aspects
- ✅ Produces a multi-panel dashboard

---

**Models used:**
- Document-level sentiment: `textattack/bert-base-uncased-SST-2`
- Aspect-based sentiment: `yangheng/deberta-v3-base-absa-v1.1`
- Aspect extraction: `spaCy en_core_web_sm` (noun/proper-noun heuristic)

**Run the notebook from top to bottom.**

## 📦 CELL 1 — Environment Setup

After this cell: **Runtime → Restart session**, then run from Cell 2.

In [ ]:
# CELL 1 — Install dependencies
import sys, subprocess

def run_pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("1/3 Removing unused torchvision...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"], check=False)

print("2/3 Installing NLP packages...")
run_pip(
    "transformers==4.46.3", "sentencepiece>=0.2.0",
    "spacy>=3.7,<4", "seaborn>=0.13",
    "matplotlib>=3.8", "pandas>=2.2,<3", "torch"
)

print("3/3 Installing spaCy English model...")
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

print("\n✅ Installation complete.")
print("🔄 IMPORTANT: Runtime → Restart session, then continue from Cell 2.")

## 📚 CELL 2 — Imports & Verification

In [ ]:
# CELL 2 — Import and verify
import sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import spacy, torch, transformers
from transformers import pipeline

# Dark theme configuration
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor": "#1a1a2e",
    "axes.edgecolor": "#3a3a5c",
    "axes.labelcolor": "#c8c8e0",
    "xtick.color": "#c8c8e0",
    "ytick.color": "#c8c8e0",
    "text.color": "#e0e0f0",
    "grid.color": "#2a2a4a",
    "grid.alpha": 0.4,
    "axes.grid": True,
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titlecolor": "#e0e0ff",
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#3a3a5c",
    "legend.labelcolor": "#c8c8e0"
})

nlp = spacy.load("en_core_web_sm")
print("✅ Environment loaded successfully!")
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"spaCy        : {spacy.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print("spaCy model  : en_core_web_sm ✅")

## 🤖 CELL 3 — Load AI Models

| Model | Purpose |
|---|---|
| `textattack/bert-base-uncased-SST-2` | Overall document-level sentiment |
| `yangheng/deberta-v3-base-absa-v1.1` | Aspect-level sentiment (ABSA) |

The ABSA model is queried with `text_pair=aspect` to predict sentiment **toward a specific aspect**.

In [ ]:
# CELL 3 — Load models
DEVICE = 0 if torch.cuda.is_available() else -1
DEVICE_LABEL = "GPU" if DEVICE == 0 else "CPU"

BERT_MODEL = "textattack/bert-base-uncased-SST-2"
print(f"Loading document-level model: {BERT_MODEL}")
sentiment_pipeline = pipeline(
    "sentiment-analysis", model=BERT_MODEL, tokenizer=BERT_MODEL, device=DEVICE
)
print(f"  ✅ Loaded  ({DEVICE_LABEL})")

ABSA_MODEL = "yangheng/deberta-v3-base-absa-v1.1"
print(f"Loading ABSA model: {ABSA_MODEL}")
absa_pipeline = pipeline(
    "text-classification", model=ABSA_MODEL, tokenizer=ABSA_MODEL, device=DEVICE
)
print(f"  ✅ Loaded  ({DEVICE_LABEL})")
print("\n🎯 Both models ready. Proceed to Cell 4.")

## ⚙️ CELL 4 — Core System Functions

1. **`extract_aspects()`** — spaCy POS-based noun extraction
2. **`analyze_aspects()`** — ABSA prediction for each aspect
3. **`overall_sentiment()`** — Document-level BERT prediction
4. **`automatic_absa()`** — End-to-end pipeline in one call

In [ ]:
# CELL 4 — Core system functions

ASPECT_STOPWORDS = {
    "thing", "way", "time", "lot", "bit", "part", "point",
    "place", "room", "day", "week", "month", "year",
    "people", "person", "man", "woman", "friend", "customer",
    "everything", "nothing", "something", "anything"
}


def extract_aspects(sentence: str) -> list:
    """
    Extract candidate aspects using spaCy POS tagging.
    Keeps unique nouns and proper nouns, excluding generic stopwords.
    Returns: list of lowercase aspect strings (deduplicated)
    """
    doc = nlp(sentence)
    aspects, seen = [], set()
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            lemma = token.lemma_.lower()
            if lemma not in seen and lemma not in ASPECT_STOPWORDS and len(lemma) > 2:
                aspects.append(token.text.lower())
                seen.add(lemma)
    return aspects


def analyze_aspects(sentence: str, aspects: list) -> pd.DataFrame:
    """
    Run ABSA for each aspect against the sentence.
    Returns: DataFrame with columns [Aspect, Sentiment, Confidence, Confidence %]
    """
    rows = []
    for aspect in aspects:
        pred = absa_pipeline(sentence, text_pair=aspect)[0]
        rows.append({
            "Aspect": aspect,
            "Sentiment": pred["label"].upper(),
            "Confidence": pred["score"]
        })
    df = pd.DataFrame(rows, columns=["Aspect", "Sentiment", "Confidence"])
    df["Confidence %"] = (df["Confidence"] * 100).round(2)
    return df


def overall_sentiment(sentence: str) -> dict:
    """
    Return overall document-level sentiment from BERT.
    Returns: dict with keys [label, score]
    """
    pred = sentiment_pipeline(sentence)[0]
    return {"label": pred["label"].upper(), "score": pred["score"]}


def automatic_absa(sentence: str) -> pd.DataFrame:
    """
    Full automatic pipeline:
    1. Extract noun/proper-noun candidates with spaCy.
    2. Run ABSA for each candidate.
    Returns: DataFrame (empty if no aspects found)
    """
    aspects = extract_aspects(sentence)
    if not aspects:
        return pd.DataFrame(columns=["Aspect", "Sentiment", "Confidence", "Confidence %"])
    return analyze_aspects(sentence, aspects)


print("✅ Core functions defined:")
print("   extract_aspects(), analyze_aspects()")
print("   overall_sentiment(), automatic_absa()")

## 🎨 CELL 5 — Visualization Functions

- `plot_aspect_bars()` — horizontal confidence bar chart coloured by sentiment
- `plot_sentiment_donut()` — positive vs negative donut chart
- `plot_confidence_heatmap()` — confidence heatmap
- `plot_pos_neg_breakdown()` — side-by-side positive/negative bars
- `plot_pipeline_flow()` — system pipeline diagram

In [ ]:
# CELL 5 — Visualization functions

POS_COLOR = "#22d3a5"
NEG_COLOR = "#f0556a"
NEU_COLOR = "#a78bfa"
BG_DARK   = "#0f0f1a"
BG_PANEL  = "#1a1a2e"
ACCENT    = "#818cf8"


def _sentiment_color(label: str) -> str:
    label = label.upper()
    if label == "POSITIVE": return POS_COLOR
    if label == "NEGATIVE": return NEG_COLOR
    return NEU_COLOR


def plot_aspect_bars(df: pd.DataFrame, title: str = "Aspect-Level Sentiment & Confidence"):
    """Horizontal bar chart: aspect vs confidence, coloured by sentiment."""
    if df.empty:
        print("No aspect data to plot.")
        return
    fig, ax = plt.subplots(figsize=(11, max(4, len(df) * 0.7 + 1)))
    colors = [_sentiment_color(s) for s in df["Sentiment"]]
    bars = ax.barh(df["Aspect"], df["Confidence"], color=colors,
                   edgecolor="#ffffff20", height=0.55)
    for bar, conf, sent in zip(bars, df["Confidence"], df["Sentiment"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{conf:.1%}  {sent.title()}", va="center", ha="left",
                fontsize=10, color=_sentiment_color(sent), fontweight="bold")
    ax.set_xlim(0, 1.35)
    ax.set_xlabel("Confidence Score")
    ax.set_ylabel("Aspect")
    ax.set_title(title, fontsize=15, pad=12, fontweight="bold")
    ax.axvline(0.5, color="#ffffff30", linestyle="--", linewidth=1)
    pos_p = mpatches.Patch(color=POS_COLOR, label="Positive")
    neg_p = mpatches.Patch(color=NEG_COLOR, label="Negative")
    neu_p = mpatches.Patch(color=NEU_COLOR, label="Neutral")
    ax.legend(handles=[pos_p, neg_p, neu_p], loc="lower right")
    plt.tight_layout()
    plt.show()


def plot_sentiment_donut(df: pd.DataFrame, title: str = "Sentiment Distribution"):
    """Donut chart: positive vs negative count."""
    if df.empty:
        print("No data to plot.")
        return
    counts = df["Sentiment"].value_counts()
    colors = [_sentiment_color(s) for s in counts.index]
    fig, ax = plt.subplots(figsize=(6, 6))
    wedges, texts, autotexts = ax.pie(
        counts.values, labels=counts.index, colors=colors,
        autopct="%1.0f%%", startangle=90,
        wedgeprops=dict(width=0.55, edgecolor=BG_DARK, linewidth=2)
    )
    for t in texts: t.set_color("#e0e0f0")
    for at in autotexts:
        at.set_color("#ffffff"); at.set_fontweight("bold"); at.set_fontsize(13)
    ax.text(0, 0, f"{counts.sum()}\nAspects", ha="center", va="center",
            fontsize=14, color="#e0e0f0", fontweight="bold")
    ax.set_title(title, fontsize=15, pad=16, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_confidence_heatmap(df: pd.DataFrame, title: str = "Aspect Confidence Heatmap"):
    """Heatmap: aspects as rows, confidence as a single column."""
    if df.empty:
        print("No data to plot.")
        return
    heat_df = df.set_index("Aspect")[["Confidence"]]
    fig, ax = plt.subplots(figsize=(5, max(3, len(df) * 0.6 + 1)))
    sns.heatmap(heat_df, annot=True, fmt=".2f", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.5, linecolor=BG_DARK,
                cbar_kws={"shrink": 0.7}, ax=ax)
    ax.set_title(title, fontsize=14, pad=10, fontweight="bold")
    ax.set_ylabel(""); ax.set_xlabel("")
    plt.tight_layout()
    plt.show()


def plot_pos_neg_breakdown(df: pd.DataFrame, title: str = "Positive vs Negative Aspects"):
    """Side-by-side bars for positive and negative aspects."""
    if df.empty:
        print("No data to plot.")
        return
    pos_df = df[df["Sentiment"] == "POSITIVE"].copy()
    neg_df = df[df["Sentiment"] == "NEGATIVE"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, max(3, max(len(pos_df), len(neg_df)) * 0.7 + 1)))

    def _draw(ax, sub_df, color, label):
        if sub_df.empty:
            ax.text(0.5, 0.5, f"No {label} aspects", ha="center", va="center",
                    transform=ax.transAxes, fontsize=13, color="#888899")
            ax.set_title(f"{label} Aspects", fontweight="bold"); ax.axis("off")
            return
        bars = ax.barh(sub_df["Aspect"], sub_df["Confidence"],
                       color=color, edgecolor="#ffffff15", height=0.5)
        for bar, c in zip(bars, sub_df["Confidence"]):
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{c:.1%}", va="center", ha="left", fontsize=10, color="#e0e0f0")
        ax.set_xlim(0, 1.2)
        ax.set_xlabel("Confidence")
        ax.set_title(f"{label} Aspects ({len(sub_df)})", fontweight="bold")

    _draw(axes[0], pos_df, POS_COLOR, "✅ Positive")
    _draw(axes[1], neg_df, NEG_COLOR, "❌ Negative")
    fig.suptitle(title, fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


def plot_pipeline_flow():
    """Diagram of the end-to-end ABSA pipeline."""
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.axis("off")
    steps = [
        ("📝 Customer\nReview",        "#22d3a5"),
        ("🔍 spaCy\nAspect Extraction","#818cf8"),
        ("🧠 DeBERTa\nABSA Model",     "#f59e0b"),
        ("🎯 Sentiment\nper Aspect",   "#f0556a"),
        ("📊 Intelligence\nDashboard", "#22d3a5"),
    ]
    xs = np.linspace(0.08, 0.92, len(steps))
    for (label, color), x in zip(steps, xs):
        ax.text(x, 0.5, label, ha="center", va="center",
                fontsize=12, fontweight="bold", color="#0f0f1a",
                bbox=dict(boxstyle="round,pad=0.6", facecolor=color,
                          edgecolor="#ffffff40", alpha=0.92))
    for x in xs[:-1]:
        ax.annotate("", xy=(x + 0.135, 0.5), xytext=(x + 0.08, 0.5),
                    arrowprops=dict(arrowstyle="->", lw=2.5, color="#ffffff60"))
    ax.set_title("Product Review Intelligence System — End-to-End Pipeline",
                 fontsize=15, fontweight="bold", pad=16)
    plt.tight_layout()
    plt.show()


print("✅ Visualization functions ready.")

## 🗺️ CELL 6 — System Pipeline Diagram

In [ ]:
# CELL 6 — Show end-to-end pipeline
plot_pipeline_flow()

## 🧪 CELL 7 — Demo: Predefined Product Reviews

Five real-world-style mixed reviews covering different categories.

In [ ]:
# CELL 7 — Demo reviews

DEMO_REVIEWS = [
    {
        "product": "Laptop",
        "review": (
            "The laptop has an excellent display and a blazing-fast processor, "
            "but the battery life is disappointing and the keyboard feels cheap."
        ),
    },
    {
        "product": "Smartphone",
        "review": (
            "The camera takes stunning photos and the screen is gorgeous, "
            "but the battery drains quickly and the price is outrageous."
        ),
    },
    {
        "product": "Restaurant",
        "review": (
            "The food was absolutely delicious and the staff were incredibly friendly, "
            "but the service was painfully slow and the noise level was unbearable."
        ),
    },
    {
        "product": "Headphones",
        "review": (
            "The sound quality is phenomenal and the noise cancellation works perfectly. "
            "However, the comfort is poor after long use and the build feels flimsy."
        ),
    },
    {
        "product": "Hotel",
        "review": (
            "The location is perfect and the view is breathtaking. "
            "Unfortunately, the Wi-Fi was unreliable and the breakfast was tasteless."
        ),
    },
]

for item in DEMO_REVIEWS:
    overall = overall_sentiment(item["review"])
    emoji = "✅" if overall["label"] == "POSITIVE" else "❌"
    aspects = extract_aspects(item["review"])
    print(f"🏷️  Product : {item['product']}")
    print(f"📝 Review  : {item['review']}")
    print(f"🌐 Overall : {emoji} {overall['label']} ({overall['score']:.1%})")
    print(f"🔍 Aspects : {aspects}")
    print("-" * 80)

## 📊 CELL 8 — Detailed Analysis: Single Review

Full ABSA pipeline with all four visualizations.
Change the index `[0]` to try a different demo review.

In [ ]:
# CELL 8 — Full analysis of one review
REVIEW = DEMO_REVIEWS[0]["review"]   # ← change index to try another

print("=" * 70)
print("📝 REVIEW")
print("=" * 70)
print(REVIEW)

# Step 1 — Overall sentiment
overall = overall_sentiment(REVIEW)
emoji = "✅" if overall["label"] == "POSITIVE" else "❌"
print(f"\n🌐 Overall BERT Sentiment: {emoji} {overall['label']} ({overall['score']:.1%})")

# Step 2 — Extract aspects
aspects = extract_aspects(REVIEW)
print(f"\n🔍 Extracted Aspects ({len(aspects)}): {aspects}")

# Step 3 — ABSA
print("\n⏳ Running aspect-level sentiment analysis...")
result_df = analyze_aspects(REVIEW, aspects)

print("\n📋 ABSA Results:")
display(result_df[["Aspect", "Sentiment", "Confidence %"]])

# Step 4 — Visualizations
plot_aspect_bars(result_df, f"{DEMO_REVIEWS[0]['product']} Review — Aspect Sentiment & Confidence")
plot_pos_neg_breakdown(result_df, f"{DEMO_REVIEWS[0]['product']} Review — Positive vs Negative")
plot_sentiment_donut(result_df, f"{DEMO_REVIEWS[0]['product']} Review — Sentiment Distribution")
plot_confidence_heatmap(result_df, f"{DEMO_REVIEWS[0]['product']} Review — Confidence Heatmap")

## 🔄 CELL 9 — Batch Analysis: All Demo Reviews

Process all five reviews and collect results into a single master DataFrame.

In [ ]:
# CELL 9 — Batch analysis

all_results = []

for item in DEMO_REVIEWS:
    print(f"\n⏳ Analysing: {item['product']}...")
    aspects = extract_aspects(item["review"])
    if not aspects:
        print("   ⚠️  No aspects found, skipping.")
        continue
    df = analyze_aspects(item["review"], aspects)
    df.insert(0, "Product", item["product"])
    all_results.append(df)
    print(f"   ✅ {len(df)} aspects analysed.")

master_df = pd.concat(all_results, ignore_index=True)

print("\n" + "=" * 70)
print("📋 MASTER RESULTS TABLE")
print("=" * 70)
display(master_df[["Product", "Aspect", "Sentiment", "Confidence %"]])

## 📈 CELL 10 — Cross-Product Comparison Charts

In [ ]:
# CELL 10 — Cross-product comparison charts

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart A: sentiment count per product
sent_counts = (
    master_df.groupby(["Product", "Sentiment"])
    .size().unstack(fill_value=0)
    .reindex(columns=["POSITIVE", "NEGATIVE", "NEUTRAL"], fill_value=0)
)
sent_counts.plot(kind="bar", ax=axes[0],
                 color=[POS_COLOR, NEG_COLOR, NEU_COLOR],
                 edgecolor="#ffffff15", width=0.6)
axes[0].set_title("Sentiment Count by Product", fontweight="bold")
axes[0].set_xlabel("Product")
axes[0].set_ylabel("Number of Aspects")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha="right")
axes[0].legend(title="Sentiment")

# Chart B: mean confidence per product
mean_conf = master_df.groupby("Product")["Confidence"].mean().sort_values(ascending=False)
bars = axes[1].bar(mean_conf.index, mean_conf.values,
                   color=ACCENT, edgecolor="#ffffff15", width=0.5)
for bar, val in zip(bars, mean_conf.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{val:.1%}", ha="center", va="bottom", fontsize=10)
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Mean ABSA Confidence by Product", fontweight="bold")
axes[1].set_xlabel("Product")
axes[1].set_ylabel("Mean Confidence")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right")

fig.suptitle("Cross-Product Comparison", fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

## 🖊️ CELL 11 — Interactive Mode

Enter **your own review** and optionally specify aspects.

- Leave aspects field **empty** → automatic extraction
- Or enter aspects **comma-separated** (e.g. `camera, battery, screen`)

In [ ]:
# CELL 11 — Interactive review analyser

print("=" * 70)
print("🎯 PRODUCT REVIEW INTELLIGENCE SYSTEM — Interactive Mode")
print("=" * 70)

review = input("\n📝 Enter your product review: ").strip()

if not review:
    print("⚠️  No review entered.")
else:
    aspects_input = input(
        "🔍 Aspects (comma-separated) or press Enter for auto-extraction: "
    ).strip()

    if aspects_input:
        aspects = [a.strip() for a in aspects_input.split(",") if a.strip()]
        mode = "Manual"
    else:
        aspects = extract_aspects(review)
        mode = "Automatic"

    print(f"\n🔍 Mode    : {mode}")
    print(f"   Aspects : {aspects}")

    if not aspects:
        print("⚠️  No aspects found. Try entering aspects manually.")
    else:
        overall = overall_sentiment(review)
        emoji = "✅" if overall["label"] == "POSITIVE" else "❌"
        print(f"\n🌐 Overall: {emoji} {overall['label']} ({overall['score']:.1%})")
        print("\n⏳ Running ABSA...")
        result_df = analyze_aspects(review, aspects)
        print("\n📋 Results:")
        display(result_df[["Aspect", "Sentiment", "Confidence %"]])
        plot_aspect_bars(result_df, "Your Review — Aspect Sentiment")
        plot_pos_neg_breakdown(result_df, "Your Review — Positive vs Negative")
        plot_sentiment_donut(result_df, "Your Review — Distribution")

## 🏆 CELL 12 — Final Intelligence Dashboard

A polished multi-panel dashboard with KPI cards.
Modify `DASHBOARD_REVIEW` and `DASHBOARD_ASPECTS` to use your own text.

In [ ]:
# CELL 12 — Final Intelligence Dashboard

DASHBOARD_REVIEW  = (
    "The food was delicious, the staff were friendly, "
    "but the service was slow and the prices were too high."
)
DASHBOARD_ASPECTS = ["food", "staff", "service", "prices"]  # [] = auto

# --- Run pipeline ---
dash_overall = overall_sentiment(DASHBOARD_REVIEW)
dash_aspects = DASHBOARD_ASPECTS if DASHBOARD_ASPECTS else extract_aspects(DASHBOARD_REVIEW)
dash_df = analyze_aspects(DASHBOARD_REVIEW, dash_aspects)
pos_count = (dash_df["Sentiment"] == "POSITIVE").sum()
neg_count = (dash_df["Sentiment"] == "NEGATIVE").sum()
avg_conf  = dash_df["Confidence"].mean()

# --- Build dashboard layout ---
fig = plt.figure(figsize=(16, 12), facecolor=BG_DARK)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# ── Title & KPI row ───────────────────────────────────────────────────────
ax_title = fig.add_subplot(gs[0, :])
ax_title.axis("off")
ax_title.text(0.5, 0.80, "🧠 Product Review Intelligence System",
              ha="center", va="center", fontsize=22, fontweight="bold",
              color="#e0e0ff", transform=ax_title.transAxes)
ax_title.text(0.5, 0.40, f'📝  "{DASHBOARD_REVIEW}"',
              ha="center", va="center", fontsize=11, color="#a0a0c0",
              style="italic", transform=ax_title.transAxes)

overall_emoji = "✅" if dash_overall["label"] == "POSITIVE" else "❌"
kpis = [
    ("Overall Sentiment",
     f"{overall_emoji} {dash_overall['label']}\n{dash_overall['score']:.1%}",
     POS_COLOR if dash_overall["label"] == "POSITIVE" else NEG_COLOR),
    ("Aspects Analysed", str(len(dash_df)), ACCENT),
    ("Positive Aspects", str(pos_count), POS_COLOR),
    ("Negative Aspects", str(neg_count), NEG_COLOR),
    ("Avg Confidence",   f"{avg_conf:.1%}", "#f59e0b"),
]
for i, (label, value, color) in enumerate(kpis):
    x = 0.06 + i * 0.195
    rect = mpatches.FancyBboxPatch(
        (x - 0.085, -0.25), 0.17, 0.22, boxstyle="round,pad=0.015",
        facecolor=color + "22", edgecolor=color, linewidth=1.5,
        transform=ax_title.transAxes, clip_on=False)
    ax_title.add_patch(rect)
    ax_title.text(x, -0.08, value, ha="center", va="center",
                  fontsize=13, fontweight="bold", color=color,
                  transform=ax_title.transAxes)
    ax_title.text(x, -0.22, label, ha="center", va="center",
                  fontsize=8.5, color="#888899", transform=ax_title.transAxes)

# ── Row 2: Horizontal bar chart (2 cols) + Donut ─────────────────────────
ax_bars = fig.add_subplot(gs[1, :2])
ax_bars.set_facecolor(BG_PANEL)
colors = [_sentiment_color(s) for s in dash_df["Sentiment"]]
h_bars = ax_bars.barh(dash_df["Aspect"], dash_df["Confidence"],
                      color=colors, edgecolor="#ffffff15", height=0.5)
for bar, conf, sent in zip(h_bars, dash_df["Confidence"], dash_df["Sentiment"]):
    ax_bars.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                 f"{conf:.1%}  {sent.title()}", va="center", ha="left",
                 fontsize=9.5, color=_sentiment_color(sent), fontweight="bold")
ax_bars.set_xlim(0, 1.4)
ax_bars.axvline(0.5, color="#ffffff25", linestyle="--", linewidth=1)
ax_bars.set_xlabel("Confidence Score")
ax_bars.set_title("Aspect-Level Sentiment & Confidence", fontweight="bold")

ax_donut = fig.add_subplot(gs[1, 2])
ax_donut.set_facecolor(BG_PANEL)
d_counts = dash_df["Sentiment"].value_counts()
d_colors = [_sentiment_color(s) for s in d_counts.index]
wedges, texts, autotexts = ax_donut.pie(
    d_counts.values, labels=d_counts.index, colors=d_colors,
    autopct="%1.0f%%", startangle=90,
    wedgeprops=dict(width=0.52, edgecolor=BG_DARK, linewidth=2))
for t in texts: t.set_color("#d0d0f0"); t.set_fontsize(9)
for at in autotexts:
    at.set_color("#ffffff"); at.set_fontweight("bold"); at.set_fontsize(11)
ax_donut.text(0, 0, f"{len(dash_df)}\nAspects", ha="center", va="center",
              fontsize=12, color="#e0e0f0", fontweight="bold")
ax_donut.set_title("Distribution", fontweight="bold")

# ── Row 3: Positive | Negative | Heatmap ─────────────────────────────────
ax_pos = fig.add_subplot(gs[2, 0])
ax_pos.set_facecolor(BG_PANEL)
pos_df = dash_df[dash_df["Sentiment"] == "POSITIVE"]
if not pos_df.empty:
    p_bars = ax_pos.barh(pos_df["Aspect"], pos_df["Confidence"],
                         color=POS_COLOR, edgecolor="#ffffff15", height=0.5)
    for bar, c in zip(p_bars, pos_df["Confidence"]):
        ax_pos.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{c:.1%}", va="center", ha="left", fontsize=9)
    ax_pos.set_xlim(0, 1.25)
    ax_pos.set_xlabel("Confidence")
else:
    ax_pos.text(0.5, 0.5, "No positive aspects", ha="center", va="center",
                color="#888899", fontsize=11)
    ax_pos.axis("off")
ax_pos.set_title(f"✅ Positive Aspects ({len(pos_df)})", color=POS_COLOR, fontweight="bold")

ax_neg = fig.add_subplot(gs[2, 1])
ax_neg.set_facecolor(BG_PANEL)
neg_df = dash_df[dash_df["Sentiment"] == "NEGATIVE"]
if not neg_df.empty:
    n_bars = ax_neg.barh(neg_df["Aspect"], neg_df["Confidence"],
                         color=NEG_COLOR, edgecolor="#ffffff15", height=0.5)
    for bar, c in zip(n_bars, neg_df["Confidence"]):
        ax_neg.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{c:.1%}", va="center", ha="left", fontsize=9)
    ax_neg.set_xlim(0, 1.25)
    ax_neg.set_xlabel("Confidence")
else:
    ax_neg.text(0.5, 0.5, "No negative aspects", ha="center", va="center",
                color="#888899", fontsize=11)
    ax_neg.axis("off")
ax_neg.set_title(f"❌ Negative Aspects ({len(neg_df)})", color=NEG_COLOR, fontweight="bold")

ax_heat = fig.add_subplot(gs[2, 2])
heat_df = dash_df.set_index("Aspect")[["Confidence"]]
sns.heatmap(heat_df, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, linecolor=BG_DARK,
            cbar_kws={"shrink": 0.75}, ax=ax_heat)
ax_heat.set_title("Confidence Heatmap", fontweight="bold")
ax_heat.set_ylabel(""); ax_heat.set_xlabel("")

plt.savefig("absa_dashboard.png", dpi=150, bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print("\n✅ Dashboard saved to absa_dashboard.png")

## 📝 CELL 13 — Text Summary Report

In [ ]:
# CELL 13 — Text summary report generator

def generate_report(review: str, aspects: list = None) -> str:
    """
    Generate a structured intelligence report for a product review.
    aspects=None → automatic extraction via spaCy.
    """
    overall = overall_sentiment(review)
    if aspects is None:
        aspects = extract_aspects(review)
    df = analyze_aspects(review, aspects) if aspects else pd.DataFrame()
    pos_df = df[df["Sentiment"] == "POSITIVE"] if not df.empty else pd.DataFrame()
    neg_df = df[df["Sentiment"] == "NEGATIVE"] if not df.empty else pd.DataFrame()

    lines = [
        "=" * 65,
        "  🧠  PRODUCT REVIEW INTELLIGENCE REPORT",
        "=" * 65,
        f"Review   : {review}",
        "-" * 65,
        f"Overall Sentiment : {overall['label']}  ({overall['score']:.1%} confidence)",
        f"Aspects Analysed  : {len(df)}",
        f"Positive Aspects  : {len(pos_df)}",
        f"Negative Aspects  : {len(neg_df)}",
    ]
    if not pos_df.empty:
        lines.append("\n✅ POSITIVE ASPECTS:")
        for _, row in pos_df.iterrows():
            lines.append(f"   • {row['Aspect']:<18} → {row['Confidence']:.1%}")
    if not neg_df.empty:
        lines.append("\n❌ NEGATIVE ASPECTS:")
        for _, row in neg_df.iterrows():
            lines.append(f"   • {row['Aspect']:<18} → {row['Confidence']:.1%}")
    if not df.empty:
        lines.append("\n📋 FULL ASPECT TABLE:")
        lines.append(f"   {'Aspect':<18} {'Sentiment':<12} {'Confidence':>10}")
        lines.append("   " + "-" * 42)
        for _, row in df.iterrows():
            lines.append(
                f"   {row['Aspect']:<18} {row['Sentiment']:<12} {row['Confidence']:>10.1%}"
            )
    lines.append("=" * 65)
    return "\n".join(lines)


report = generate_report(DASHBOARD_REVIEW, aspects=DASHBOARD_ASPECTS)
print(report)

with open("absa_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("\n✅ Report saved to absa_report.txt")

## 🎓 CELL 14 — Classroom Assessment

### Conceptual Questions

1. What is the difference between document-level and aspect-level sentiment?
2. Why does the ABSA model receive `text_pair=aspect`?
3. How does spaCy extract candidate aspects in this system?
4. What does **confidence** represent in the context of a classifier?
5. Why might automatic noun extraction miss important aspects?

---

### Coding Challenge

> **"The camera takes amazing photos, but the battery drains quickly and the screen is too dim."**

Expected aspects: `camera`, `photos`, `battery`, `screen`

In [ ]:
# CELL 14 — Classroom coding challenge

challenge_review  = (
    "The camera takes amazing photos, but the battery drains quickly "
    "and the screen is too dim."
)
challenge_aspects = ["camera", "photos", "battery", "screen"]

print("Review:")
print(challenge_review)
print()
print("Before running, predict the sentiment for each aspect:")
for a in challenge_aspects:
    print(f"   {a:12} → ?")
print()

challenge_df = analyze_aspects(challenge_review, challenge_aspects)
print("Model predictions:")
display(challenge_df[["Aspect", "Sentiment", "Confidence %"]])

plot_aspect_bars(challenge_df, "Challenge Review — Aspect Sentiment")
plot_pos_neg_breakdown(challenge_df, "Challenge Review — Positive vs Negative")

## 🏁 CELL 15 — Final Summary

### What this system does

| Step | Component | Description |
|---|---|---|
| 1 | **Input** | Accepts any product / service review |
| 2 | **Extraction** | spaCy POS tagger finds noun/proper-noun candidates |
| 3 | **ABSA** | DeBERTa model predicts sentiment per aspect |
| 4 | **Confidence** | Softmax probability from the classifier |
| 5 | **Visualization** | Bar charts, donut chart, heatmap |
| 6 | **Dashboard** | Multi-panel summary with KPI cards |
| 7 | **Report** | Structured text summary saved to file |

---

### Key takeaway

> Standard sentiment analysis asks: *What is the overall sentiment?*
>
> ABSA asks: *What is the sentiment **toward each specific aspect**?*

```
"I love the camera but hate the battery."

camera  → POSITIVE  (high confidence)
battery → NEGATIVE  (high confidence)
```

This distinction is the foundation of practical NLP review intelligence.